In [0]:
%pip install statsmodels pmdarima

In [0]:
import mlflow, pickle
import pandas as pd
import numpy as np
from mlflow.tracking import MlflowClient

client = MlflowClient()
experiment = client.get_experiment_by_name("/Users/santhoshnagendrarajan@gmail.com/weather-sarima")

# Cell 1: Find the best run by RMSE
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.rmse ASC"]
)
best_run = runs[0]
best_run_id = best_run.info.run_id
print(f"Best run: {best_run.data.tags.get('mlflow.runName')}  RMSE: {best_run.data.metrics['rmse']:.4f}")



In [0]:
# Cell 2: Note - model was logged as artifact, not MLflow model
# The sarima_model.pkl was logged via log_artifact(), not log_model()
# so it cannot be registered to the model registry.
# Cell 3 downloads it directly as an artifact for inference.
print(f"Best model artifact available at: runs:/{best_run_id}/sarima_model.pkl")
print("Model will be loaded directly from artifacts for inference.")



In [0]:
# Cell 3: Batch inference — load model and forecast next 30 days
artifact_path = client.download_artifacts(best_run_id, "sarima_model.pkl", "/tmp/")
with open("/tmp/sarima_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

forecast_steps = 30
forecast = loaded_model.forecast(steps=forecast_steps)
forecast_conf = loaded_model.get_forecast(steps=forecast_steps).conf_int()

last_date = pd.to_datetime("2023-12-31")
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=forecast_steps)

forecast_pdf = pd.DataFrame({
    "date": future_dates,
    "forecast_temp": forecast.values,
    "lower_ci": forecast_conf.iloc[:, 0].values,
    "upper_ci": forecast_conf.iloc[:, 1].values,
    "model_run_id": best_run_id
})

# Write to Gold Delta table
df_forecast = spark.createDataFrame(forecast_pdf)
df_forecast.write.format("delta").mode("overwrite").saveAsTable("weather_forecast_gold")
display(df_forecast)